In [1]:
import os
import json
import csv
from dataclasses import dataclass, asdict
from typing import Dict, Tuple, Optional, Literal, Any

import numpy as np
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm

import datasets
import unet
from SplitNet import SplitNet
import prof_unet


# -----------------------------------------------------------------------------
# Reproducibility / device
# -----------------------------------------------------------------------------

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

ModelType = Literal["splitnet_attn", "splitnet", "unet", "attn_unet", "prof_unet"]
DatasetMode = Literal["border", "border_pressure", "fixed"]
TrainingMode = Literal["physics_limited", "baseline_full"]


@dataclass
class ExperimentConfig:
    model_type: ModelType = "splitnet_attn"
    dataset_mode: DatasetMode = "border"

    # physics_limited:
    #   Uses Limited datasets: (input_sample, sparse_target, mask)
    #   MSE trains only on mask points.
    #   Darcy loss is applied over the full model output.
    #
    # baseline_full:
    #   Uses Full datasets: (input_sample, full_target)
    #   MSE trains over the whole output.
    #   Darcy loss must be 0.
    training_mode: TrainingMode = "physics_limited"

    mse_weight: float = 1.0
    darcy_weight: float = 1.0

    epochs: int = 250
    batch_size: int = 8
    lr: float = 1e-3

    channels: str = "KP"
    train_sims_path: str = "../train_sims.npy"
    val_sims_path: str = "../val_sims.npy"
    sim_max_exclusive: Optional[int] = 500

    save_prefix: Optional[str] = None
    save_best: bool = True
    save_final: bool = True

    # Folder where experiment data tables/checkpoints are saved.
    # If save_prefix is given, files are saved using that prefix.
    # If save_prefix is None, save_dir/run_name are used instead.
    save_dir: str = "minimum_info/results"
    run_name: Optional[str] = None

    # For physics_limited, this chooses how masked MSE is computed.
    # "old_zeroed" matches your old script:
    #     crit(out * mask, label * mask)
    # This divides by the whole image size, including zeros outside the mask.
    # "true_masked" divides only by the number of masked pixels.
    mask_loss_style: str = "old_zeroed"

    # If None, run_experiment chooses a sensible default:
    #     physics_limited -> val_supervised_mse
    #     baseline_full    -> val_total_mse
    best_metric: Optional[str] = None

    # Optional dataset overrides, e.g.
    # dataset_kwargs={"points_per_side": 5, "radius": 3, "steps": (0, 200)}
    dataset_kwargs: Optional[Dict[str, Any]] = None


# -----------------------------------------------------------------------------
# Darcy utilities
# -----------------------------------------------------------------------------


def darcy_residual_map(out: torch.Tensor) -> torch.Tensor:
    """
    Computes div(K * grad(P)) for output tensors whose channel order is K, P, ...

    Expected shape: [B, C, H, W]
    Requires at least 2 output channels: channel 0 = K, channel 1 = P.

    Returns residual map with shape [B, 1, H, W].
    """
    if out.shape[1] < 2:
        raise ValueError("Darcy residual requires at least 2 output channels: K and P.")

    k = out[:, 0:1]
    p = out[:, 1:2]

    p_y, p_x = torch.gradient(p, dim=(-2, -1))

    flux_y = k * p_y
    flux_x = k * p_x

    div_y = torch.gradient(flux_y, spacing=(1,), dim=(-2,))[0]
    div_x = torch.gradient(flux_x, spacing=(1,), dim=(-1,))[0]

    return div_y + div_x


def darcy_loss_from_output(out: torch.Tensor) -> torch.Tensor:
    """
    Physics loss used by this experiment runner.

    This matches the old training.py pattern: when using masked/limited training,
    the MSE is masked, but the Darcy loss is still computed over the full output.
    """
    return (darcy_residual_map(out) ** 2).mean()


# -----------------------------------------------------------------------------
# Model factory
# -----------------------------------------------------------------------------


def make_model(model_type: ModelType = "splitnet_attn", channels: str = "KP") -> nn.Module:
    """
    Builds a model matching the selected channel setup.

    For channels='KP', all models output 2 channels: K and P.
    SplitNet already outputs K and P, so it is only compatible with channels='KP'.
    """
    model_type = model_type.lower()

    if channels == "all":
        num_channels = 3
    elif channels == "KP":
        num_channels = 2
    elif channels in ["K", "P", "phi"]:
        num_channels = 1
    else:
        raise ValueError("channels must be 'all', 'KP', 'K', 'P', or 'phi'.")

    if model_type == "splitnet_attn":
        if channels != "KP":
            raise ValueError("SplitNet is designed for channels='KP'.")
        return SplitNet(attn=True).to(DEVICE)

    if model_type == "splitnet":
        if channels != "KP":
            raise ValueError("SplitNet is designed for channels='KP'.")
        return SplitNet(attn=False).to(DEVICE)

    if model_type == "unet":
        return unet.SmallUnet(channels=num_channels).to(DEVICE)

    if model_type == "attn_unet":
        return unet.AttnUnet(channels=num_channels).to(DEVICE)
    
    if model_type == "prof_unet":
        return prof_unet.ResizedUNet(
            in_channels=num_channels,
            num_classes=num_channels,
            size=256,
        ).to(DEVICE)

    raise ValueError(f"Unknown model_type: {model_type}")


# -----------------------------------------------------------------------------
# Dataset / loader factory
# -----------------------------------------------------------------------------


def _dataset_class(dataset_mode: DatasetMode, training_mode: TrainingMode):
    """
    Chooses the correct dataset class for the experiment.

    training_mode='physics_limited':
        Uses Limited datasets.
        These return (sample, target, mask).
        The target is sparse/limited, and the mask tells us where supervised MSE is allowed.

    training_mode='baseline_full':
        Uses Full datasets.
        These return (sample, full_target).
        Supervised MSE is computed over the full output.
    """
    if training_mode == "physics_limited":
        if dataset_mode == "border":
            return datasets.BorderDenseDatasetLimited
        if dataset_mode == "border_pressure":
            return datasets.BorderThinPressureGradientDatasetLimited
        if dataset_mode == "fixed":
            return datasets.FixedDenseDatasetLimited
        if dataset_mode == "random":
            return datasets.RandomDenseDatasetLimited

    if training_mode == "baseline_full":
        if dataset_mode == "border":
            return datasets.BorderDenseDatasetFull
        if dataset_mode == "border_pressure":
            return datasets.BorderDensePressureGradientDatasetFull
        if dataset_mode == "fixed":
            return datasets.FixedDenseDatasetFull

    raise ValueError(
        "Invalid dataset options. dataset_mode must be 'border', 'fixed', or 'random'; "
        "training_mode must be 'physics_limited' or 'baseline_full'."
    )


def load_sim_ids(path: str, sim_max_exclusive: Optional[int] = 500) -> np.ndarray:
    sims = np.load(path)
    if sim_max_exclusive is not None:
        sims = sims[sims < sim_max_exclusive]
    return sims

class ChannelSelectDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, channels="KP"):
        self.base_dataset = base_dataset
        self.channels = channels

    def _idx(self):
        if self.channels == "all":
            return [0, 1, 2]
        if self.channels == "KP":
            return [0, 1]
        if self.channels == "K":
            return [0]
        if self.channels == "P":
            return [1]
        if self.channels == "phi":
            return [2]
        raise ValueError("Bad channels")

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        chans = self._idx()

        if len(item) == 3:
            feat, label, mask = item
            return feat[chans], label[chans], mask

        feat, label = item
        return feat[chans], label[chans]


def make_loaders(config: ExperimentConfig) -> Tuple[DataLoader, DataLoader]:
    train_sims = load_sim_ids(config.train_sims_path, config.sim_max_exclusive)
    val_sims = load_sim_ids(config.val_sims_path, config.sim_max_exclusive)

    dataset_cls = _dataset_class(config.dataset_mode, config.training_mode)

    kwargs = dict(config.dataset_kwargs or {})
    kwargs["channels"] = config.channels

    train_data = dataset_cls(train_sims, **kwargs)
    val_data = dataset_cls(val_sims, **kwargs)

    train_data = ChannelSelectDataset(train_data, channels=config.channels)
    val_data = ChannelSelectDataset(val_data, channels=config.channels)

    train_loader = DataLoader(train_data, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=config.batch_size, shuffle=False)

    return train_loader, val_loader


# -----------------------------------------------------------------------------
# Saving helpers
# -----------------------------------------------------------------------------


def safe_float(x):
    """Converts tensors/numpy values to JSON/CSV-friendly Python floats."""
    if isinstance(x, torch.Tensor):
        return float(x.detach().cpu().item())
    if isinstance(x, np.generic):
        return float(x)
    return x


def make_run_prefix(config: ExperimentConfig) -> str:
    """
    Returns the path prefix for all saved files from a run.

    Example prefix:
        minimum_info/results/fixed_physics_limited_splitnet_attn_darcy_1p0
    """
    if config.save_prefix:
        return config.save_prefix

    if config.run_name:
        name = config.run_name
    else:
        safe_w = str(config.darcy_weight).replace(".", "p")
        name = f"{config.dataset_mode}_{config.training_mode}_{config.model_type}_darcy_{safe_w}"

    return os.path.join(config.save_dir, name)


def ensure_parent_dir(path_prefix: str):
    folder = os.path.dirname(path_prefix)
    if folder:
        os.makedirs(folder, exist_ok=True)


def save_json(path: str, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def save_history_csv(path: str, history: Dict[str, Any]):
    """
    Saves epoch-by-epoch curves to CSV.

    This is the main file to use later for plotting train/val curves.
    """
    curve_keys = [
        "train_loss_used",
        "train_total_mse",
        "train_supervised_mse",
        "train_mask_mse",
        "train_nonmask_mse",
        "train_darcy",
        "val_total_mse",
        "val_supervised_mse",
        "val_mask_mse",
        "val_nonmask_mse",
        "val_darcy",
    ]

    n_epochs = len(history["train_loss_used"])

    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch"] + curve_keys)

        for i in range(n_epochs):
            row = [i + 1]
            for key in curve_keys:
                values = history.get(key, [])
                row.append(values[i] if i < len(values) else "")
            writer.writerow(row)


def make_run_summary(history: Dict[str, Any], config: ExperimentConfig, best_epoch: int, best_val_score: float) -> Dict[str, Any]:
    """
    Saves one compact summary per run.
    Useful for comparing runs without loading every curve.
    """
    def best(key):
        values = history[key]
        return min(values) if len(values) else None

    def final(key):
        values = history[key]
        return values[-1] if len(values) else None

    summary = {
        "run_name": config.run_name,
        "model_type": config.model_type,
        "dataset_mode": config.dataset_mode,
        "training_mode": config.training_mode,
        "channels": config.channels,
        "mse_weight": config.mse_weight,
        "darcy_weight": config.darcy_weight,
        "mask_loss_style": config.mask_loss_style,
        "epochs": config.epochs,
        "batch_size": config.batch_size,
        "lr": config.lr,
        "best_epoch": best_epoch,
        "best_val_score_used": best_val_score,
        "best_train_loss_used": best("train_loss_used"),
        "best_val_total_mse": best("val_total_mse"),
        "best_val_supervised_mse": best("val_supervised_mse"),
        "best_val_mask_mse": best("val_mask_mse"),
        "best_val_nonmask_mse": best("val_nonmask_mse"),
        "best_val_darcy": best("val_darcy"),
        "final_train_loss_used": final("train_loss_used"),
        "final_val_total_mse": final("val_total_mse"),
        "final_val_supervised_mse": final("val_supervised_mse"),
        "final_val_mask_mse": final("val_mask_mse"),
        "final_val_nonmask_mse": final("val_nonmask_mse"),
        "final_val_darcy": final("val_darcy"),
        "config": asdict(config),
    }

    return summary


def save_run_outputs(path_prefix: str, model: nn.Module, history: Dict[str, Any], config: ExperimentConfig, best_epoch: int, best_val_score: float):
    """
    Saves all analysis-ready outputs for a single experiment run.

    Files created:
        *_history.csv       epoch curves for plotting/comparison
        *_history.pt        full Python history dict
        *_summary.json      compact final/best metrics + config
        *_final_state.pt    model weights, if enabled
    """
    ensure_parent_dir(path_prefix)

    save_history_csv(f"{path_prefix}_history.csv", history)
    torch.save(history, f"{path_prefix}_history.pt")

    summary = make_run_summary(history, config, best_epoch, best_val_score)
    save_json(f"{path_prefix}_summary.json", summary)

    if config.save_final:
        torch.save(model.state_dict(), f"{path_prefix}_final_state.pt")


# -----------------------------------------------------------------------------
# Loss / metric helpers
# -----------------------------------------------------------------------------


def align_channels(label: torch.Tensor, out: torch.Tensor) -> torch.Tensor:
    """
    Keeps label channels compatible with model output channels.
    Example: if a dataset gives 3 channels but the model outputs KP only, keep K and P.
    """
    if label.shape[1] == out.shape[1]:
        return label
    if label.shape[1] > out.shape[1]:
        return label[:, : out.shape[1]]
    raise ValueError(f"Label has {label.shape[1]} channels but output has {out.shape[1]} channels.")


def expand_mask_for_channels(mask: torch.Tensor, out: torch.Tensor) -> torch.Tensor:
    if mask.dim() == 3:
        mask = mask.unsqueeze(1)
    return mask.expand(-1, out.shape[1], -1, -1)


def mse_on_region(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """
    True masked MSE. This divides by number of masked pixels, not by whole image size.
    """
    mask = mask.float()
    diff2 = ((pred - target) ** 2) * mask
    denom = mask.sum()
    if denom.item() == 0:
        return torch.tensor(0.0, device=pred.device)
    return diff2.sum() / denom


def supervised_loss(
    out: torch.Tensor,
    label: torch.Tensor,
    mask: Optional[torch.Tensor],
    training_mode: TrainingMode,
    crit: nn.Module,
    mask_loss_style: str = "old_zeroed",
) -> torch.Tensor:
    """
    baseline_full:
        MSE over the whole output.

    physics_limited:
        MSE only over mask points.

        mask_loss_style='old_zeroed' matches your old script exactly:
            crit(out * mask, label * mask)

        mask_loss_style='true_masked' computes a mathematically true masked MSE:
            sum((out-label)^2 over mask) / number_of_mask_pixels
    """
    label = align_channels(label, out)

    if training_mode == "baseline_full":
        return crit(out, label)

    if training_mode == "physics_limited":
        if mask is None:
            raise ValueError("physics_limited mode requires a mask.")
        mask_c = expand_mask_for_channels(mask.bool(), out).float()

        if mask_loss_style == "old_zeroed":
            return crit(out * mask_c, label * mask_c)

        if mask_loss_style == "true_masked":
            return mse_on_region(out, label, mask_c)

        raise ValueError("mask_loss_style must be 'old_zeroed' or 'true_masked'.")

    raise ValueError(f"Unknown training_mode: {training_mode}")


def unpack_batch(batch, training_mode: TrainingMode):
    if training_mode == "physics_limited":
        feat, label, mask = batch
        return feat, label, mask
    if training_mode == "baseline_full":
        feat, label = batch
        return feat, label, None
    raise ValueError(f"Unknown training_mode: {training_mode}")


# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------


def evaluate_loader(model: nn.Module, loader: DataLoader, config: ExperimentConfig, crit: nn.Module) -> Dict[str, float]:
    model.eval()

    total_mse = 0.0
    supervised_mse = 0.0
    mask_mse = 0.0
    nonmask_mse = 0.0
    darcy = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch in loader:
            feat, label, mask = unpack_batch(batch, config.training_mode)
            feat = feat.to(DEVICE)
            label = label.to(DEVICE)
            mask = mask.to(DEVICE).bool() if mask is not None else None

            out = model(feat)
            label = align_channels(label, out)

            total_mse += crit(out, label).item()
            supervised_mse += supervised_loss(
                out, label, mask, config.training_mode, crit, config.mask_loss_style
            ).item()
            darcy += darcy_loss_from_output(out).item()

            if mask is not None:
                mask_c = expand_mask_for_channels(mask, out)
                nonmask_c = ~mask_c
                mask_mse += mse_on_region(out, label, mask_c).item()
                nonmask_mse += mse_on_region(out, label, nonmask_c).item()
            else:
                mask_mse += float("nan")
                nonmask_mse += float("nan")

            n_batches += 1

    return {
        "total_mse": total_mse / n_batches,
        "supervised_mse": supervised_mse / n_batches,
        "mask_mse": mask_mse / n_batches,
        "nonmask_mse": nonmask_mse / n_batches,
        "darcy": darcy / n_batches,
    }


# -----------------------------------------------------------------------------
# Training
# -----------------------------------------------------------------------------


def run_experiment(**kwargs):
    """
    Main experiment entry point.

    Physics run:
        model, history = run_experiment(
            model_type="splitnet_attn",
            dataset_mode="fixed",
            training_mode="physics_limited",
            mse_weight=1.0,
            darcy_weight=0.1,
            epochs=250,
            save_prefix="minimum_info/fixed_splitattn_darcy_0p1"
        )

    No-physics baseline:
        model, history = run_experiment(
            model_type="splitnet_attn",
            dataset_mode="fixed",
            training_mode="baseline_full",
            mse_weight=1.0,
            darcy_weight=0.0,
            epochs=250,
            save_prefix="minimum_info/fixed_splitattn_baseline"
        )
    """
    config = ExperimentConfig(**kwargs)

    if config.training_mode == "baseline_full" and config.darcy_weight != 0:
        raise ValueError(
            "baseline_full is the no-physics baseline. Set darcy_weight=0.0, "
            "or use training_mode='physics_limited'."
        )

    if config.training_mode == "physics_limited" and config.darcy_weight == 0:
        print(
            "Warning: physics_limited with darcy_weight=0.0 uses limited/masked MSE but no physics. "
            "For the no-physics full-MSE baseline, use training_mode='baseline_full'."
        )

    path_prefix = make_run_prefix(config)
    ensure_parent_dir(path_prefix)

    model = make_model(config.model_type, config.channels)
    optimizer = Adam(model.parameters(), lr=config.lr)
    crit = nn.MSELoss()

    train_loader, val_loader = make_loaders(config)

    history = {
        "train_loss_used": [],
        "train_total_mse": [],
        "train_supervised_mse": [],
        "train_mask_mse": [],
        "train_nonmask_mse": [],
        "train_darcy": [],
        "val_total_mse": [],
        "val_supervised_mse": [],
        "val_mask_mse": [],
        "val_nonmask_mse": [],
        "val_darcy": [],
        "config": asdict(config),
    }

    best_val_score = float("inf")
    best_epoch = 0

    for epoch in tqdm(range(1, config.epochs + 1)):
        model.train()
        epoch_loss = 0.0
        n_batches = 0

        for batch in train_loader:
            feat, label, mask = unpack_batch(batch, config.training_mode)
            feat = feat.to(DEVICE)
            label = label.to(DEVICE)
            mask = mask.to(DEVICE).bool() if mask is not None else None

            optimizer.zero_grad()

            out = model(feat)
            label = align_channels(label, out)

            mse = supervised_loss(out, label, mask, config.training_mode, crit, config.mask_loss_style)
            physics = darcy_loss_from_output(out)

            loss = config.mse_weight * mse + config.darcy_weight * physics
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        history["train_loss_used"].append(epoch_loss / n_batches)

        train_metrics = evaluate_loader(model, train_loader, config, crit)
        val_metrics = evaluate_loader(model, val_loader, config, crit)

        history["train_total_mse"].append(train_metrics["total_mse"])
        history["train_supervised_mse"].append(train_metrics["supervised_mse"])
        history["train_mask_mse"].append(train_metrics["mask_mse"])
        history["train_nonmask_mse"].append(train_metrics["nonmask_mse"])
        history["train_darcy"].append(train_metrics["darcy"])

        history["val_total_mse"].append(val_metrics["total_mse"])
        history["val_supervised_mse"].append(val_metrics["supervised_mse"])
        history["val_mask_mse"].append(val_metrics["mask_mse"])
        history["val_nonmask_mse"].append(val_metrics["nonmask_mse"])
        history["val_darcy"].append(val_metrics["darcy"])

        # Model selection metric:
        # In physics_limited, total_mse is misleading because the label is sparse/zeroed.
        # supervised_mse is the validation version of the actual training MSE.
        if config.best_metric is not None:
            val_score = val_metrics[config.best_metric]
        elif config.training_mode == "physics_limited":
            val_score = val_metrics["supervised_mse"]
        else:
            val_score = val_metrics["total_mse"]

        if val_score < best_val_score:
            best_val_score = val_score
            best_epoch = epoch

            if config.save_best:
                ensure_parent_dir(path_prefix)
                torch.save(model.state_dict(), f"{path_prefix}_best_state.pt")

    metric_name = config.best_metric or ("val_supervised_mse" if config.training_mode == "physics_limited" else "val_total_mse")
    print(f"Best epoch: {best_epoch}, best {metric_name}: {best_val_score:.6f}")

    save_run_outputs(path_prefix, model, history, config, best_epoch, best_val_score)

    return model, history


# -----------------------------------------------------------------------------
# Convenience helper for physics weight sweeps
# -----------------------------------------------------------------------------


def save_sweep_summary_csv(path: str, results: Dict[Any, Dict[str, Any]]):
    """
    Saves one CSV row per Darcy weight in a scale sweep.
    This is the main file for later drawing a Darcy weight curve.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)

    fieldnames = [
        "darcy_weight",
        "best_val_total_mse",
        "best_val_supervised_mse",
        "final_val_total_mse",
        "final_val_supervised_mse",
        "final_val_darcy",
        "best_epoch_by_supervised",
        "best_epoch_by_total",
    ]

    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for w, r in results.items():
            hist = r["history"]
            best_epoch_by_supervised = int(np.argmin(hist["val_supervised_mse"])) + 1
            best_epoch_by_total = int(np.argmin(hist["val_total_mse"])) + 1

            writer.writerow({
                "darcy_weight": w,
                "best_val_total_mse": min(hist["val_total_mse"]),
                "best_val_supervised_mse": min(hist["val_supervised_mse"]),
                "final_val_total_mse": hist["val_total_mse"][-1],
                "final_val_supervised_mse": hist["val_supervised_mse"][-1],
                "final_val_darcy": hist["val_darcy"][-1],
                "best_epoch_by_supervised": best_epoch_by_supervised,
                "best_epoch_by_total": best_epoch_by_total,
            })


def run_darcy_weight_sweep(
    darcy_weights,
    model_type: ModelType = "splitnet_attn",
    dataset_mode: DatasetMode = "fixed",
    epochs: int = 250,
    base_save_dir: str = "minimum_info",
    **kwargs,
):
    """
    Runs physics_limited experiments across multiple Darcy weights.

    Saves:
        one history CSV per individual run
        one summary JSON per individual run
        one sweep_summary.csv for the full curve
    """
    results = {}

    for w in darcy_weights:
        safe_w = str(w).replace(".", "p")
        run_name = f"{dataset_mode}_physics_limited_{model_type}_darcy_{safe_w}"

        model, history = run_experiment(
            model_type=model_type,
            dataset_mode=dataset_mode,
            training_mode="physics_limited",
            mse_weight=1.0,
            darcy_weight=float(w),
            epochs=epochs,
            save_dir=base_save_dir,
            run_name=run_name,
            **kwargs,
        )

        results[w] = {
            "model": model,
            "history": history,
            "best_val_total_mse": min(history["val_total_mse"]),
            "best_val_supervised_mse": min(history["val_supervised_mse"]),
            "final_val_total_mse": history["val_total_mse"][-1],
            "final_val_supervised_mse": history["val_supervised_mse"][-1],
            "final_val_darcy": history["val_darcy"][-1],
        }

    save_sweep_summary_csv(
        os.path.join(base_save_dir, f"{dataset_mode}_{model_type}_sweep_summary.csv"),
        results,
    )

    return results


# -----------------------------------------------------------------------------
# Experiment plans / example runs
# -----------------------------------------------------------------------------


def run_physics_scale_tests(
    model_type: ModelType = "splitnet_attn",
    dataset_modes=("fixed", "border", "random"),
    epochs: int = 75,
    base_save_dir: str = "minimum_info/scale_tests",
):
    """
    Wide first-pass sweep to figure out the useful Darcy-weight scale.

    Your MSE may be much larger than Darcy loss, so this intentionally tests
    several orders of magnitude. These are shorter runs meant to identify a
    reasonable range before doing 250-epoch final comparisons.
    """
    darcy_weights = [
        0.0,
        0.001,
        0.01,
        0.1,
        1.0,
        10.0,
    ]

    all_results = {}

    for dataset_mode in dataset_modes:
        print(f"===== Scale test: {model_type}, {dataset_mode} =====")
        results = run_darcy_weight_sweep(
            darcy_weights,
            model_type=model_type,
            dataset_mode=dataset_mode,
            epochs=epochs,
            base_save_dir=f"{base_save_dir}/{dataset_mode}",
        )
        all_results[dataset_mode] = results

    return all_results


def run_final_comparison_grid(
    darcy_weight: float,
    epochs: int = 250,
    dataset_modes=("fixed", "border", "random"),
    model_types=("splitnet_attn", "splitnet", "unet", "attn_unet"),
    base_save_dir: str = "minimum_info/final_grid",
):
    """
    Final comparison after choosing a Darcy weight.

    For each dataset/mask option and each model type, this runs:
        1. Physics model:
           limited/masked MSE + Darcy loss over full output
        2. No-physics baseline:
           full-image MSE + no Darcy loss

    Note:
        baseline_full is unavailable for dataset_mode='random' unless you add
        a RandomDenseDatasetFull class to datasets.py. The code skips that case.
    """
    results = {}

    for dataset_mode in dataset_modes:
        for model_type in model_types:
            key_base = f"{dataset_mode}_{model_type}"
            results[key_base] = {}

            # Physics run: Limited dataset + masked MSE + Darcy loss.
            physics_run_name = (
                f"{dataset_mode}_physics_limited_{model_type}_darcy_"
                f"{str(darcy_weight).replace('.', 'p')}"
            )

            print(f"===== Physics run: {dataset_mode}, {model_type}, darcy={darcy_weight} =====")
            model_physics, hist_physics = run_experiment(
                model_type=model_type,
                dataset_mode=dataset_mode,
                training_mode="physics_limited",
                mse_weight=1.0,
                darcy_weight=darcy_weight,
                epochs=epochs,
                save_dir=f"{base_save_dir}/{dataset_mode}",
                run_name=physics_run_name,
            )

            results[key_base]["physics_limited"] = {
                "model": model_physics,
                "history": hist_physics,
            }

            # Baseline run: Full dataset + full-image MSE + no Darcy.
            if dataset_mode == "random":
                print(
                    "Skipping random baseline_full because RandomDenseDatasetFull "
                    "does not exist in the current datasets.py."
                )
                continue

            baseline_run_name = f"{dataset_mode}_baseline_full_{model_type}_nodarcy"

            print(f"===== Baseline run: {dataset_mode}, {model_type} =====")
            model_base, hist_base = run_experiment(
                model_type=model_type,
                dataset_mode=dataset_mode,
                training_mode="baseline_full",
                mse_weight=1.0,
                darcy_weight=0.0,
                epochs=epochs,
                save_dir=f"{base_save_dir}/{dataset_mode}",
                run_name=baseline_run_name,
            )

            results[key_base]["baseline_full"] = {
                "model": model_base,
                "history": hist_base,
            }

    return results


def summarize_results(results):
    """
    Prints a compact summary from either run_physics_scale_tests or
    run_final_comparison_grid outputs.
    """
    print("===== Summary =====")

    for key, value in results.items():
        print(f"--- {key} ---")

        # Case 1: scale sweep result, keyed by Darcy weight.
        if all(isinstance(k, float) or isinstance(k, int) for k in value.keys()):
            for w, r in value.items():
                print(
                    f"darcy_weight={w:<10} | "
                    f"best_val_total_mse={r['best_val_total_mse']:.6f} | "
                    f"best_val_supervised_mse={r['best_val_supervised_mse']:.6f} | "
                    f"final_val_darcy={r['final_val_darcy']:.6f}"
                )

        # Case 2: final grid result, keyed by run type.
        else:
            for run_name, r in value.items():
                hist = r["history"]
                print(
                    f"{run_name:<18} | "
                    f"best_val_total_mse={min(hist['val_total_mse']):.6f} | "
                    f"best_val_supervised_mse={min(hist['val_supervised_mse']):.6f} | "
                    f"final_val_darcy={hist['val_darcy'][-1]:.6f}"
                )


if __name__ == "__main__":
    # -------------------------------------------------------------------------
    # STEP 1: Find a reasonable Darcy-weight scale.
    # -------------------------------------------------------------------------
    # Start with shorter runs across a very wide range.
    # You are looking for weights where Darcy improves/regularizes the output
    # without making MSE much worse.

    # scale_results = run_physics_scale_tests(
    #     model_type="attn_unet",
    #     dataset_modes=("fixed", "border_pressure",),
    #     epochs=75,
    #     base_save_dir="recent_analysis_unet/physics_tests",
    # )
    # summarize_results(scale_results)

    # -------------------------------------------------------------------------
    # STEP 2: After choosing a Darcy weight, run the final comparison grid.
    # -------------------------------------------------------------------------
    # Example: if the scale test suggests 100.0 is reasonable, use that below.

    # final_results = run_final_comparison_grid(
    #     darcy_weight=100.0,
    #     epochs=250,
    #     dataset_modes=("fixed", "border", "random"),
    #     model_types=("splitnet_attn", "splitnet", "unet", "attn_unet"),
    #     base_save_dir="minimum_info/final_grid",
    # )
    # summarize_results(final_results)
    pass


In [ ]:
# scale_results = run_physics_scale_tests(
#     model_type="prof_unet",
#     dataset_modes=("border",),
#     epochs=75,
#     base_save_dir="recent_analysis_prof_unet/physics_tests",
# )
# summarize_results(scale_results)

===== Scale test: prof_unet, border =====


100%|██████████| 75/75 [13:36<00:00, 10.89s/it]


Best epoch: 30, best val_supervised_mse: 0.001106


100%|██████████| 75/75 [13:36<00:00, 10.89s/it]


Best epoch: 32, best val_supervised_mse: 0.001051


...

100%|██████████| 75/75 [13:45<00:00, 11.01s/it]


Best epoch: 58, best val_supervised_mse: 0.001275


100%|██████████| 75/75 [13:17<00:00, 10.64s/it]


Best epoch: 50, best val_supervised_mse: 0.001250


 19%|█▊        | 14/75 [02:34<11:21, 11.18s/it]

In [2]:
# scale_results = run_physics_scale_tests(
#     model_type="splitnet",
#     dataset_modes=("fixed", "border_pressure", "border"),
#     epochs=75,
#     base_save_dir="recent_analysis_split_na/physics_tests",
# )
# summarize_results(scale_results)

scale_results = run_physics_scale_tests(
    model_type="unet",
    dataset_modes=("border_pressure", "border"),
    epochs=75,
    base_save_dir="recent_analysis_unet_na/physics_tests",
)
summarize_results(scale_results)

===== Scale test: unet, border_pressure =====


  0%|          | 0/75 [00:03<?, ?it/s]


KeyboardInterrupt: 

In [17]:
def run_core_baseline_full(
    model_types=("splitnet_attn", "attn_unet", "prof_unet"),
    dataset_modes=("fixed", "border",),
    epochs=20,
    save_dir="baseline_full/final",
):
    results = {}

    for dataset_mode in dataset_modes:
        for model_type in model_types:
            run_name = f"{dataset_mode}_{model_type}_baseline_full_nodarcy"

            print(f"\n===== Running {run_name} =====")

            model, hist = run_experiment(
                model_type=model_type,
                dataset_mode=dataset_mode,
                training_mode="baseline_full",
                darcy_weight=0.0,
                epochs=epochs,
                sim_max_exclusive=100,
                save_dir=save_dir,
                run_name=run_name,
            )

            results[run_name] = {"model": model, "history": hist}

    return results

run_core_baseline_full()


===== Running fixed_splitnet_attn_baseline_full_nodarcy =====


100%|██████████| 20/20 [2:13:23<00:00, 400.18s/it]  


Best epoch: 4, best val_total_mse: 0.006091

===== Running fixed_attn_unet_baseline_full_nodarcy =====


100%|██████████| 20/20 [1:45:54<00:00, 317.75s/it]


Best epoch: 16, best val_total_mse: 0.007745

===== Running fixed_prof_unet_baseline_full_nodarcy =====


...

100%|██████████| 20/20 [1:01:12<00:00, 183.63s/it]


Best epoch: 6, best val_total_mse: 0.006514

===== Running border_prof_unet_baseline_full_nodarcy =====


100%|██████████| 20/20 [2:50:13<00:00, 510.66s/it]  

Best epoch: 3, best val_total_mse: 0.010106


{'fixed_splitnet_attn_baseline_full_nodarcy': {'model': SplitNet(
    (K_encode): Encode(
      (c1): TwoConv(
        (seq): Sequential(
          (0): Conv2d(1, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU()
          (3): Conv2d(4, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (4): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (5): ReLU()
        )
      )
      (d1): Downsample(
        (conv): TwoConv(
          (seq): Sequential(
            (0): Conv2d(4, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU()
            (3): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (4): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True

In [5]:


def run_core_darcy_models(
    model_darcy_choices=None,
    dataset_modes=("fixed",),
    epochs=20,
    sim_max_exclusive=100,
    save_dir="official_darcy/final",
):
    """
    Runs official physics-limited Darcy models.

    model_darcy_choices maps each model type to the Darcy weights
    you want to officially train.
    """
    if model_darcy_choices is None:
        model_darcy_choices = {
            # "splitnet_attn": [1.0, 10.0],
            "prof_unet": [1.0, 10.0],
            "attn_unet": [1.0, 10.0],
        }

    results = {}

    for dataset_mode in dataset_modes:
        for model_type, darcy_weights in model_darcy_choices.items():
            for darcy_weight in darcy_weights:
                safe_w = str(darcy_weight).replace(".", "p")
                run_name = (
                    f"{dataset_mode}_physics_limited_"
                    f"{model_type}_darcy_{safe_w}"
                )

                print(f"\n===== Running {run_name} =====")

                model, hist = run_experiment(
                    model_type=model_type,
                    dataset_mode=dataset_mode,
                    training_mode="physics_limited",
                    mse_weight=1.0,
                    darcy_weight=darcy_weight,
                    epochs=epochs,
                    sim_max_exclusive=sim_max_exclusive,
                    save_dir=save_dir,
                    run_name=run_name,
                )

                results[run_name] = {
                    "model": model,
                    "history": hist,
                }

    return results


official_darcy_results = run_core_darcy_models()


===== Running fixed_physics_limited_prof_unet_darcy_1p0 =====


100%|██████████| 20/20 [3:22:06<00:00, 606.32s/it]  


Best epoch: 14, best val_supervised_mse: 0.000643

===== Running fixed_physics_limited_prof_unet_darcy_10p0 =====


100%|██████████| 20/20 [3:19:43<00:00, 599.17s/it]  


Best epoch: 9, best val_supervised_mse: 0.000732

===== Running fixed_physics_limited_attn_unet_darcy_1p0 =====


100%|██████████| 20/20 [1:39:51<00:00, 299.59s/it]


Best epoch: 17, best val_supervised_mse: 0.000124

===== Running fixed_physics_limited_attn_unet_darcy_10p0 =====


 45%|████▌     | 9/20 [49:08<1:00:03, 327.61s/it]


KeyboardInterrupt: 

In [ ]:


def run_core_darcy_models(
    model_darcy_choices=None,
    dataset_modes=("border",),
    epochs=20,
    sim_max_exclusive=100,
    save_dir="official_darcy/final",
):
    """
    Runs official physics-limited Darcy models.

    model_darcy_choices maps each model type to the Darcy weights
    you want to officially train.
    """
    ["splitnet_attn", "splitnet", "unet", "attn_unet", "prof_unet"]
    if model_darcy_choices is None:
        model_darcy_choices = {
            # "splitnet_attn": [5.0, 0.1],
            # "prof_unet": [5.0, 0.1],
            # "attn_unet": [0.1],
            "splitnet": [1.0, 10.0],
            "unet": [5.0, 0.1, 1.0, 10.0],

        }

    results = {}

    for dataset_mode in dataset_modes:
        for model_type, darcy_weights in model_darcy_choices.items():
            for darcy_weight in darcy_weights:
                safe_w = str(darcy_weight).replace(".", "p")
                run_name = (
                    f"{dataset_mode}_physics_limited_"
                    f"{model_type}_darcy_{safe_w}"
                )

                print(f"\n===== Running {run_name} =====")

                model, hist = run_experiment(
                    model_type=model_type,
                    dataset_mode=dataset_mode,
                    training_mode="physics_limited",
                    mse_weight=1.0,
                    darcy_weight=darcy_weight,
                    epochs=epochs,
                    sim_max_exclusive=sim_max_exclusive,
                    save_dir=save_dir,
                    run_name=run_name,
                )

                results[run_name] = {
                    "model": model,
                    "history": hist,
                }

    return results


official_darcy_results = run_core_darcy_models()




















def run_core_darcy_models(
    model_darcy_choices=None,
    dataset_modes=("fixed",),
    epochs=20,
    sim_max_exclusive=100,
    save_dir="official_darcy/final",
):
    """
    Runs official physics-limited Darcy models.

    model_darcy_choices maps each model type to the Darcy weights
    you want to officially train.
    """
    if model_darcy_choices is None:
        model_darcy_choices = {
            "splitnet_attn": [5.0, 0.1, 1.0, 10.0],
            "prof_unet": [5.0, 0.1, 1.0, 10.0],
            "attn_unet": [5.0, 0.1, 1.0, 10.0],
            "splitnet": [5.0, 0.1, 1.0, 10.0],
            "unet": [5.0, 0.1, 1.0, 10.0],

        }

    results = {}

    for dataset_mode in dataset_modes:
        for model_type, darcy_weights in model_darcy_choices.items():
            for darcy_weight in darcy_weights:
                safe_w = str(darcy_weight).replace(".", "p")
                run_name = (
                    f"{dataset_mode}_physics_limited_"
                    f"{model_type}_darcy_{safe_w}"
                )

                print(f"\n===== Running {run_name} =====")

                model, hist = run_experiment(
                    model_type=model_type,
                    dataset_mode=dataset_mode,
                    training_mode="physics_limited",
                    mse_weight=1.0,
                    darcy_weight=darcy_weight,
                    epochs=epochs,
                    sim_max_exclusive=sim_max_exclusive,
                    save_dir=save_dir,
                    run_name=run_name,
                )

                results[run_name] = {
                    "model": model,
                    "history": hist,
                }

    return results


official_darcy_results = run_core_darcy_models()


===== Running border_physics_limited_splitnet_darcy_1p0 =====


100%|██████████| 20/20 [1:04:15<00:00, 192.79s/it]


Best epoch: 15, best val_supervised_mse: 0.000558

===== Running border_physics_limited_splitnet_darcy_10p0 =====


100%|██████████| 20/20 [1:03:29<00:00, 190.46s/it]


Best epoch: 14, best val_supervised_mse: 0.000649

===== Running border_physics_limited_unet_darcy_5p0 =====


...

Best epoch: 15, best val_supervised_mse: 0.000598

===== Running border_physics_limited_unet_darcy_10p0 =====


100%|██████████| 20/20 [47:25<00:00, 142.26s/it]


Best epoch: 17, best val_supervised_mse: 0.000631

===== Running fixed_physics_limited_attn_unet_darcy_10p0 =====


100%|██████████| 20/20 [1:34:32<00:00, 283.63s/it]

Best epoch: 17, best val_supervised_mse: 0.000122


In [ ]:
def run_data_amount_comparison(
    sim_amounts=(10, 25, 50),
    dataset_modes=("fixed", "border"),
    epochs=20,
    save_dir="data_amount_tests",
):
    """
    Tests whether Darcy helps more when less training data is available.

    For each sim amount, this runs:
      1. physics_limited + Darcy
      2. baseline_full + no Darcy

    Kept intentionally small so it can run overnight.
    """

    physics_choices = {
        "splitnet_attn": [1.0, 10.0],
        "attn_unet": [1.0, 10.0],
        "prof_unet": [1.0, 10.0],
    }

    baseline_models = [
        "splitnet_attn",
        "attn_unet",
        "prof_unet",
    ]

    results = {}

    for sim_max in sim_amounts:
        for dataset_mode in dataset_modes:

            # --------------------------------------------------
            # Physics / Darcy models
            # --------------------------------------------------
            for model_type, darcy_weights in physics_choices.items():
                for darcy_weight in darcy_weights:
                    safe_w = str(darcy_weight).replace(".", "p")

                    run_name = (
                        f"{dataset_mode}_{model_type}_"
                        f"physics_limited_darcy_{safe_w}_sims_{sim_max}"
                    )

                    print(f"\n===== Running {run_name} =====")

                    model, hist = run_experiment(
                        model_type=model_type,
                        dataset_mode=dataset_mode,
                        training_mode="physics_limited",
                        mse_weight=1.0,
                        darcy_weight=darcy_weight,
                        epochs=epochs,
                        sim_max_exclusive=sim_max,
                        save_dir=f"{save_dir}/physics_limited",
                        run_name=run_name,
                    )

                    results[run_name] = {
                        "model": model,
                        "history": hist,
                    }

            # --------------------------------------------------
            # Full baseline / no physics models
            # --------------------------------------------------
            for model_type in baseline_models:
                run_name = (
                    f"{dataset_mode}_{model_type}_"
                    f"baseline_full_nodarcy_sims_{sim_max}"
                )

                print(f"\n===== Running {run_name} =====")

                model, hist = run_experiment(
                    model_type=model_type,
                    dataset_mode=dataset_mode,
                    training_mode="baseline_full",
                    mse_weight=1.0,
                    darcy_weight=0.0,
                    epochs=epochs,
                    sim_max_exclusive=sim_max,
                    save_dir=f"{save_dir}/baseline_full",
                    run_name=run_name,
                )

                results[run_name] = {
                    "model": model,
                    "history": hist,
                }

    return results


data_amount_results = run_data_amount_comparison()


===== Running fixed_splitnet_attn_physics_limited_darcy_1p0_sims_10 =====


100%|██████████| 20/20 [11:58<00:00, 35.94s/it]


Best epoch: 20, best val_supervised_mse: 0.000102

===== Running fixed_splitnet_attn_physics_limited_darcy_10p0_sims_10 =====


100%|██████████| 20/20 [11:53<00:00, 35.69s/it]


Best epoch: 12, best val_supervised_mse: 0.000325

===== Running fixed_attn_unet_physics_limited_darcy_1p0_sims_10 =====


...

100%|██████████| 20/20 [22:16<00:00, 66.81s/it]


Best epoch: 18, best val_supervised_mse: 0.000785

===== Running border_attn_unet_physics_limited_darcy_1p0_sims_25 =====


100%|██████████| 20/20 [15:39<00:00, 46.99s/it]


Best epoch: 20, best val_supervised_mse: 0.000680

===== Running border_attn_unet_physics_limited_darcy_10p0_sims_25 =====


 75%|███████▌  | 15/20 [12:21<04:19, 51.82s/it]